# 4.2 · Comparación final y NeuralHydrology

**Tiempo estimado:** 30 min.

**Objetivos.**

1. Comparar **SARIMAX (S2)** vs **LightGBM (S3)** vs **LSTM (S4)** con el mismo split temporal y métricas hidrológicas.
2. Discutir en qué casos gana cada uno.
3. Conocer **NeuralHydrology** como referencia del DL hidrológico moderno.

## Mini-intro (10 min)

**Honestidad metodológica.** Para comparar de verdad:

- Mismo periodo de evaluación (test 2018-2020).
- Mismas métricas (NSE, KGE, error pico).
- Si el modelo es estocástico (LSTM): **media de varias semillas**, no una sola.
- Reportar también qué información usa cada uno (¿lluvia futura oracular? ¿no?).

**NeuralHydrology** (Kratzert et al., JKU Linz):

- Librería específica de DL hidrológico, con LSTMs especiales (EA-LSTM, MTS-LSTM).
- Entrenamiento *multi-basin* (cientos de cuencas a la vez) → ha mostrado superar modelos calibrados localmente.
- Configuración por YAML, no código. Se ejecuta como CLI.

Aquí solo la presentamos; instalarla y entrenarla excede una sesión presencial.

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Input
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
from skforecast.recursive import ForecasterRecursive

from cst import datos as ud

plt.rcParams.update({"figure.figsize": (10, 3.4), "axes.grid": True, "grid.alpha": 0.3})
keras.utils.set_random_seed(0)

## 1 · Reproducimos los tres modelos

Versiones simplificadas (sin walk-forward para que la celda vuelva rápido).

In [ ]:
caudal = ud.cargar_caudal_genil(source="CEDEX")
lluvia = ud.cargar_lluvia_genil_diaria(fecha_inicio="2010-01-01", fecha_fin="2020-12-31")
df_d = pd.DataFrame({"caudal": caudal, "lluvia": lluvia}).loc["2011":"2020"]
df_d = df_d.asfreq("D").interpolate("linear", limit=7).dropna()
df_m = df_d.resample("MS").agg({"caudal": "mean", "lluvia": "sum"}).dropna()

split = pd.Timestamp("2018-01-01")
print(f"Diario: train {len(df_d.loc[:split]):,}  test {len(df_d.loc[split:]):,}")
print(f"Mensual: train {len(df_m.loc[:split]):,}  test {len(df_m.loc[split:]):,}")

### 1.a SARIMAX mensual (1 paso, sin reajuste)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

tr_m = df_m.loc[:split].iloc[:-1]
te_m = df_m.loc[split:]

exog_tr = tr_m["lluvia"].shift(1).fillna(method="bfill").values.reshape(-1, 1)
m_sx = SARIMAX(
    tr_m["caudal"],
    exog=exog_tr,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False, maxiter=100)
exog_te = te_m["lluvia"].values.reshape(-1, 1)
pred_sx_m = m_sx.forecast(steps=len(te_m), exog=exog_te)
print(f"SARIMAX listo. RMSE mensual: {np.sqrt(((te_m['caudal'] - pred_sx_m) ** 2).mean()):.3f}")

### 1.b LightGBM diario (skforecast, h=1)

In [ ]:
def exog_diaria(df):
    out = pd.DataFrame(index=df.index)
    for w in (3, 7, 14, 30):
        out[f"p_acum{w}d"] = df["lluvia"].rolling(w).sum().shift(1)
    idx = out.index
    out["sin_an"] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    out["cos_an"] = np.cos(2 * np.pi * idx.dayofyear / 365.25)
    return out.asfreq("D")


X_d = exog_diaria(df_d).dropna().asfreq("D")
y_d = df_d["caudal"].loc[X_d.index].asfreq("D")
y_tr_d, y_te_d = y_d.loc[:split].iloc[:-1].asfreq("D"), y_d.loc[split:].asfreq("D")
X_tr_d, X_te_d = X_d.loc[y_tr_d.index].asfreq("D"), X_d.loc[y_te_d.index].asfreq("D")

f_lgb = ForecasterRecursive(
    regressor=lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=31, n_jobs=-1, verbose=-1, random_state=0
    ),
    lags=[1, 2, 3, 7, 14, 30, 90, 365],
)
f_lgb.fit(y=y_tr_d, exog=X_tr_d)
pred_lgb_d = f_lgb.predict(steps=len(y_te_d), exog=X_te_d)
print(f"LightGBM listo. RMSE diario: {np.sqrt(((y_te_d - pred_lgb_d) ** 2).mean()):.3f}")

### 1.c LSTM diario (h=1)

In [ ]:
VENTANA = 60
FEATS = ["caudal", "lluvia"]


def crear_ventanas(df, features, target, ventana, horizonte=1):
    X, y, idx = [], [], []
    val_f = df[features].values
    val_t = df[target].values
    fechas = df.index
    for i in range(len(val_f) - ventana - horizonte + 1):
        X.append(val_f[i : i + ventana])
        y.append(val_t[i + ventana + horizonte - 1])
        idx.append(fechas[i + ventana + horizonte - 1])
    return np.asarray(X), np.asarray(y), pd.DatetimeIndex(idx)


X_all, y_all, idx_all = crear_ventanas(df_d, FEATS, "caudal", VENTANA)
mask_tr = idx_all < pd.Timestamp("2016-01-01")
mask_va = (idx_all >= pd.Timestamp("2016-01-01")) & (idx_all < split)
mask_te = idx_all >= split

sX = StandardScaler().fit(X_all[mask_tr].reshape(-1, len(FEATS)))
sy = StandardScaler().fit(y_all[mask_tr].reshape(-1, 1))


def tx(X, y):
    return (
        sX.transform(X.reshape(-1, len(FEATS))).reshape(X.shape),
        sy.transform(y.reshape(-1, 1)).flatten(),
    )


Xtr, ytr = tx(X_all[mask_tr], y_all[mask_tr])
Xva, yva = tx(X_all[mask_va], y_all[mask_va])
Xte, _ = tx(X_all[mask_te], y_all[mask_te])

model = Sequential([Input(shape=(VENTANA, len(FEATS))), LSTM(32), Dropout(0.2), Dense(1)])
model.compile(optimizer="adam", loss="mse")
model.fit(
    Xtr,
    ytr,
    validation_data=(Xva, yva),
    epochs=25,
    batch_size=64,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=0,
)

pred_lstm_d = sy.inverse_transform(model.predict(Xte, verbose=0)).flatten()
obs_lstm_d = y_all[mask_te]
idx_lstm = idx_all[mask_te]
print(f"LSTM listo. RMSE diario: {np.sqrt(((obs_lstm_d - pred_lstm_d) ** 2).mean()):.3f}")

## 2 · Tabla comparativa final

Métricas hidrológicas sobre el test 2018-2020. Para comparar a la misma frecuencia, agregamos diario a mensual.

In [ ]:
def nse(o, s):
    o, s = np.asarray(o), np.asarray(s)
    return 1 - np.sum((o - s) ** 2) / np.sum((o - o.mean()) ** 2)


def kge(o, s):
    o, s = np.asarray(o), np.asarray(s)
    r = np.corrcoef(o, s)[0, 1]
    a = s.std() / o.std()
    b = s.mean() / o.mean()
    return 1 - np.sqrt((r - 1) ** 2 + (a - 1) ** 2 + (b - 1) ** 2)


# SARIMAX ya está en mensual
obs_sx = te_m["caudal"].values

# Agregamos LGB y LSTM diarios a mensual para comparar 1:1
lgb_mes = pd.Series(pred_lgb_d.values, index=pred_lgb_d.index).resample("MS").mean()
lstm_mes = pd.Series(pred_lstm_d, index=idx_lstm).resample("MS").mean()
obs_mes = pd.Series(obs_lstm_d, index=idx_lstm).resample("MS").mean()

lgb_mes = lgb_mes.reindex(obs_mes.index)
lstm_mes = lstm_mes.reindex(obs_mes.index)

filas = []
for nombre, pred_obs in [
    ("SARIMAX (mensual)", (obs_sx, pred_sx_m.values)),
    ("LightGBM (diario→mes)", (obs_mes.values, lgb_mes.values)),
    ("LSTM (diario→mes)", (obs_mes.values, lstm_mes.values)),
]:
    o, s = pred_obs
    mask = ~np.isnan(s)
    filas.append(
        {
            "modelo": nombre,
            "NSE": round(nse(o[mask], s[mask]), 3),
            "KGE": round(kge(o[mask], s[mask]), 3),
            "RMSE": round(float(np.sqrt(((o[mask] - s[mask]) ** 2).mean())), 3),
            "Err. pico": round(float(s[mask].max() - o[mask].max()), 3),
        }
    )
pd.DataFrame(filas).set_index("modelo")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(obs_mes.index, obs_mes.values, color="black", lw=1.5, label="observado mensual")
ax.plot(te_m.index, pred_sx_m, color="#c2410c", lw=1, ls="--", label="SARIMAX")
ax.plot(lgb_mes.index, lgb_mes.values, color="#16a34a", lw=1, ls="--", label="LightGBM")
ax.plot(lstm_mes.index, lstm_mes.values, color="#2563eb", lw=1, ls="--", label="LSTM")
ax.set_ylabel("Q mensual (m³/s)")
ax.legend(ncol=4)
ax.set_title("Comparación a frecuencia mensual — test 2018-2020")
plt.tight_layout()

## 3 · Lectura — ¿quién gana?

**Observaciones típicas en cuencas reguladas:**

- **SARIMAX** captura bien la media estacional. Empata o gana en NSE mensual cuando hay pocos datos.
- **LightGBM** brilla a frecuencia **diaria** y con muchas features. Su agregado mensual es competitivo.
- **LSTM** necesita **mucho dato** (> 10 años) para ser estable. Aquí, con 18 años, suele empatar a LightGBM o quedarse algo por detrás. En cuencas con > 30 años de datos densos y multivariate, supera al resto.

**¿Cuándo escalar a DL?**

- Cuando el ML clásico satura.
- Cuando entrenamos **muchas cuencas a la vez** (transfer / multi-task learning).
- Cuando necesitamos predicción **multi-step** larga con dependencias temporales complejas.

## 4 · NeuralHydrology — demo conceptual

[NeuralHydrology](https://neuralhydrology.readthedocs.io/) es el estándar de DL hidrológico (Kratzert et al., JKU Linz). **No se aprende a programar con ella**: se configura por YAML.

### Configuración mínima (config.yml)

```yaml
experiment_name: genil_lstm_demo
model: cudalstm
hidden_size: 64
seq_length: 365             # un año hacia atrás
predict_last_n: 1

target_variables: [QObs(mm/d)]
dynamic_inputs:  [precipitation, temperature, srad]
static_attributes:  [area, elevation, p_mean, baseflow_index]

dataset: camels_es                       # versión española de CAMELS
data_dir: /path/to/CAMELS-ES
train_basin_file: train_basins.txt
train_start_date: 01/10/1980
train_end_date:   30/09/2010
test_start_date:  01/10/2010
test_end_date:    30/09/2020

epochs: 30
batch_size: 256
```

### Comandos

```bash
pip install neuralhydrology
nh-run train --config-file config.yml
nh-run evaluate --run-dir runs/<exp>
```

### Por qué importa

- Soporta entrenamiento **multi-basin** (cientos de cuencas a la vez) — la fuente principal de su mejora.
- Implementa LSTMs especializadas para hidrología (EA-LSTM, MTS-LSTM).
- Es la referencia de la mayoría de papers de DL hidrológico actuales.

Si quieres seguir profundizando tras el curso: empieza por [este tutorial oficial](https://neuralhydrology.readthedocs.io/en/latest/tutorials/introduction.html).

## 5 · Ejercicios

1. **Ensemble.** Combina las predicciones SARIMAX + LightGBM + LSTM con media simple. ¿Bate al mejor individual?
2. **Frecuencia diaria.** Compara LightGBM vs LSTM a frecuencia diaria (sin agregar a mensual). ¿Cambia el ranking?
3. **Reto.** Reentrena LSTM con `seq_length=365` (un año). ¿Mejora el NSE? ¿A qué coste?
4. **CAMELS-ES.** Descarga CAMELS-ES desde Zenodo y monta un experimento NeuralHydrology con 5 cuencas. (Tarea de proyecto extendido — fuera de la sesión.)